Denna fil skapar din träningsdata. Vi ökar antalet ticks till 100 och ser till att vi har en jämn fördelning av attacker


Cell 1: Bibliotek och Setup
Rad 1-6: Importerar nödvändiga verktyg för datahantering, nätverksanrop och databas.

Rad 9-10: Definierar vilken AI-modell vi pratar med och var den finns på din dator

In [1]:
import json
import random
import requests
import pandas as pd
from datetime import datetime
import chromadb
from chromadb.utils import embedding_functions

# Konfiguration för Ollama
MODEL = "gemma4:e4b" 
OLLAMA_API = "http://localhost:11434/api/generate"

Cell 2: Simulatorn & RAG (Hjärnan)

Simulator: Skapar en miljö där vi kan "skruva upp" trafiken för att simulera attacker.

RAG: Laddar din säkerhetshandbok i en sökbar databas så att agenten kan läsa reglerna.

In [2]:
# Simulator-klass för att skapa nätverkstrafik
class NetworkSimulator:
    def __init__(self):
        self.devices = {
            "10.0.0.1": {"name": "Huvudserver", "traffic": 50, "notes": "Normal"},
            "192.168.1.100": {"name": "IoT-Kamera", "traffic": 5, "notes": "Idle"}
        }
    def get_telemetry(self):
        for ip in self.devices:
            self.devices[ip]["traffic"] = random.randint(10, 60)
            self.devices[ip]["notes"] = "Stable"
        return self.devices
    def trigger_attack(self, type):
        if type == "DDoS":
            self.devices["10.0.0.1"]["traffic"] = random.randint(700, 1000)
            self.devices["10.0.0.1"]["notes"] = "Massive incoming requests"
        elif type == "Exfiltration":
            self.devices["192.168.1.100"]["traffic"] = 8
            self.devices["192.168.1.100"]["notes"] = "Connecting to external IP: 45.12.3.1"

# Initiera RAG-databasen
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="policy_db")
policies = [
    "IoT 192.168.1.100 får ej prata med externa IP (45.x.x.x).",
    "Server 10.0.0.1 trafik > 500 tyder på DDoS.",
    "Normal trafik för alla enheter ska ligga under 100."
]
collection.add(documents=policies, ids=[f"id{i}" for i in range(len(policies))])

Cell 3: Insamlingsloopen (Skapar 100 rader data)

ask_agent: Kombinerar nätverksdata med handboken och frågar Gemma om råd.

Loopen: Kör 100 gånger. Tack vare tick % 10 och tick % 15 får vi ca 15-20 attacker totalt, vilket räcker för att stratify ska fungera i nästa steg.

In [3]:
def ask_agent(data):
    # RAG: Hämta relevant policy
    res = collection.query(query_texts=[str(data)], n_results=1)
    context = res['documents'][0][0]
    prompt = f"Policy: {context}\nData: {json.dumps(data)}\nSvara i JSON: {{'decision': 'isolate'/'none', 'reason': 'varför'}}"
    resp = requests.post(OLLAMA_API, json={"model": MODEL, "prompt": prompt, "stream": False, "format": "json"})
    return json.loads(resp.json()['response'])

sim = NetworkSimulator()
logs = []

for tick in range(1, 101):
    # Skapa attacker var 10:e och var 15:e tick för att få tillräckligt med '1'-etts (stratify-vänligt)
    if tick % 10 == 0: sim.trigger_attack("DDoS")
    elif tick % 15 == 0: sim.trigger_attack("Exfiltration")
    
    current_data = sim.get_telemetry()
    agent_resp = ask_agent(current_data)
    
    # Spara resultatet. Vi sätter 'is_attack' till 1 om agenten valde isolate.
    logs.append({
        "metrics": current_data.copy(),
        "target": 1 if agent_resp['decision'] == 'isolate' else 0,
        "policy_used": agent_resp.get('reason', 'N/A')
    })
    if tick % 10 == 0: print(f"Tick {tick} klart...")

pd.DataFrame(logs).to_json("network_audit_log.jsonl", orient="records", lines=True)

Tick 10 klart...
Tick 20 klart...
Tick 30 klart...
Tick 40 klart...
Tick 50 klart...
Tick 60 klart...
Tick 70 klart...
Tick 80 klart...
Tick 90 klart...
Tick 100 klart...
